# Gymnasium の基本 API とランダム方策のベースライン

**出典**　機械学習 4 分野の教科書 — 第 12 章 4-B 方策ベース・発展・実践 ／ 07 実装：Gymnasium と Stable-Baselines3

**このノートのパス**　`ml-code/12_policy_gradient/01_gymnasium_basics.ipynb`

**このノートで確かめること**
`reset` / `step` の 5 つ組を確認し、**ランダム方策の得点**を測る。
**この数字が、学習したエージェントを比べる相手になる。**

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `gymnasium` |
| 実行時間の目安 | 約 10 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q gymnasium

In [3]:
import gymnasium as gym

env = gym.make("CartPole-v1")           # 台車の上の棒を倒さないよう左右に動かす課題
print(env.observation_space)            # Box(4,)  位置・速度・角度・角速度
print(env.action_space)                 # Discrete(2)  左 / 右

obs, info = env.reset(seed=42)
total = 0.0
for _ in range(500):
    action = env.action_space.sample()               # ランダム方策
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    if terminated or truncated:
        break
print(f"ランダム方策の得点: {total}")   # だいたい 20 前後（倒れるのが早い）
env.close()

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Discrete(2)
ランダム方策の得点: 13.0


### 1 回では分からない — 強化学習の評価は必ず複数エピソードの平均で

In [4]:
import numpy as np

def evaluate_random(env_id="CartPole-v1", n_episodes=100, seed=42):
    env = gym.make(env_id)
    rs = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed + i)
        done, total = False, 0.0
        while not done:
            obs, r, term, trunc, _ = env.step(env.action_space.sample())
            total += r; done = term or trunc
        rs.append(total)
    env.close()
    return np.array(rs)


rs = evaluate_random()
print(f"ランダム方策 {len(rs)} エピソード: "
      f"平均 {rs.mean():.1f} ± {rs.std():.1f}  (最小 {rs.min():.0f} / 最大 {rs.max():.0f})")
print("\n※ 1 エピソードだけ見ると最大 {:.0f} という数字も出る。".format(rs.max()))
print("   学習結果を「1 回走らせて良かった」で判断してはいけない理由がここにある。")

ランダム方策 100 エピソード: 平均 22.6 ± 10.5  (最小 9 / 最大 59)

※ 1 エピソードだけ見ると最大 59 という数字も出る。
   学習結果を「1 回走らせて良かった」で判断してはいけない理由がここにある。


### 5 つ組の意味を確認する

In [5]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("obs      :", obs, "← 位置・速度・角度・角速度")
obs, reward, terminated, truncated, info = env.step(0)
print("reward   :", reward, "← CartPole は 1 ステップ立っていれば +1")
print("terminated:", terminated, "← 棒が倒れた / 台車が枠外に出た（本当の終端）")
print("truncated :", truncated, "← 500 ステップの時間切れ（終端ではない）")
env.close()

obs      : [ 0.01369617 -0.02302133 -0.04590265 -0.04834723] ← 位置・速度・角度・角速度
reward   : 1.0 ← CartPole は 1 ステップ立っていれば +1
terminated: False ← 棒が倒れた / 台車が枠外に出た（本当の終端）
truncated : False ← 500 ステップの時間切れ（終端ではない）


## 読み取りどころ

- **ランダム方策の得点（20 前後）が、すべての比較の出発点になる。**教師あり学習でベースラインを置くのと同じ役割。
- ⚠️ **`terminated` と `truncated` を区別する。**学習コードでブートストラップを切るのは `terminated` のときだけ。混同すると「長く粘ると損」と学習してしまう。
- ⚠️ **分散が非常に大きい。**ランダム方策でも 1 エピソードなら 60 点以上が出ることがある。**学習の成否を 1 回の実行で判断してはいけない。**

得点 500 が CartPole-v1 の上限（`truncated` する長さ）。次のノートの PPO はここに届く。